# Assignment 04: Blood Pressure Follow-Up

`README.md` gives each task's steps and checkpoints; the headings here match its numbering. Run each cell with `Shift+Enter`. When you finish, click **Restart**, then **Run All**, and run `python3 check_assignment.py` in the terminal.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("pyarrow:", pyarrow.__version__)

DATA_PATH = Path("data") / "bp_followup.csv"
HOME_PATH = Path("data") / "home_bp.csv"
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
LOADED_PATH = OUTPUT_DIR / "bp_loaded.csv"
SUMMARY_PATH = OUTPUT_DIR / "visit_summary.csv"
COUNTS_PATH = OUTPUT_DIR / "clinic_counts.csv"
FOLLOWUP_PATH = OUTPUT_DIR / "followup_priority.csv"
PARQUET_PATH = OUTPUT_DIR / "followup_priority.parquet"
GAP_PATH = OUTPUT_DIR / "white_coat_gap.csv"

print("data files found:", DATA_PATH.exists() and HOME_PATH.exists())

NumPy: 2.3.3
pandas: 3.0.5
pyarrow: 25.0.0
data files found: True


## Task 1: Put the cells in running order

### 1.1 Reorder the cells

The next two code cells are out of order: the first uses `vials_on_hand`, which the second defines. Move the whole producer cell above the dependent cell. Do not copy the definition into another cell.

In [2]:
vials_on_hand = 3
print("vials_on_hand:", vials_on_hand)

vials_on_hand: 3


In [3]:
doses_available = vials_on_hand * 10
print("doses_available:", doses_available)

doses_available: 30


### 1.2 Explain the repair

Explain the difference between the order the cells appear in and the order they ran, why the first cell can print a result even while it sits above the cell it needs, why output saved under a cell is not proof that the notebook works now, and what you changed and how Restart and Run All shows that it worked.

The Python kernel remembers every variable defined based on the order you run them, not the order they appear on the page. So even if the information the first cell needs is in the cell after it, if you run the second cell first, the kernel will have the defined information saved so that when you go back to run the first cell, it will be able to access the information from the kernel's periwinkle working memory. Outputs are thus products of the order in which cells were run, not necessarily reflective of a working order of commands. Without switching the order of the cells, if I tried to Run All, the notebook would run from top to bottom and the first cell would not have access to the information it needs to run. By switching the order of the cells, I ensure that if the kernel is restarted and I Run All, the now second cell will always have the information it needs to run.

## Task 2: Load the blood pressure export

### 2.1 Look at the raw file

Run this cell as it is: it prints the file and reads it with no options.

In [4]:
print(open(DATA_PATH).read())
raw = pd.read_csv(DATA_PATH)
print("shape:", raw.shape)
raw.head(3)

patient_id;clinic;age;sbp_baseline;sbp_week4;sbp_week8;coordinator_note
id;text;years;mmHg;mmHg;mmHg;text
P101;North;58;152;146;138;Started lisinopril at baseline
P102;South;64;146;140;135;
P103;West;47;138;134;131;Diet counseling only
P104;North;71;164;-999;150;Missed week 4 visit
P105;West;55;158;150;147;
P106;East;62;149;141;136;Dose increased at week 4
P107;North;49;140;137;133;
P108;South;68;171;160;152;Referred to cardiology
P109;West;60;155;149;-999;Moved before week 8
P110;East;53;160;152;145;
P111;East;66;-999;145;139;Baseline cuff too small
P112;North;45;132;130;128;
P113;East;70;168;158;155;Home monitor issued
P114;North;57;145;143;137;

shape: (15, 1)


,patient_id;clinic;age;sbp_baseline;sbp_week4;sbp_week8;coordinator_note
0,id;text;years;mmHg;mmHg;mmHg;text
1,P101;North;58;152;146;138;Started lisinopril a...
2,P102;South;64;146;140;135;


### 2.2 Read with options and save

> **Checkpoint: `output/bp_loaded.csv`**
> 14 rows under the header `patient_id,clinic,age,sbp_baseline,sbp_week4,sbp_week8`.

In [21]:
# TODO: read DATA_PATH with sep, skiprows, usecols, na_values, and index_col
bp = pd.read_csv(DATA_PATH, 
                 sep=";", 
                 skiprows=[1], 
                 usecols=['patient_id', 'clinic', 'age', 'sbp_baseline', 'sbp_week4', 'sbp_week8'],
                 na_values="-999", 
                 index_col="patient_id") 

print("shape:", bp.shape)
print(bp.dtypes)
display(bp.head())

# TODO: display patient P104's row with .loc, then the first three rows with .iloc
display(bp.loc["P104"])
display(bp.iloc[0:3])

# TODO: save bp to LOADED_PATH, keeping its patient_id index
bp.to_csv(LOADED_PATH)

shape: (14, 5)
clinic              str
age               int64
sbp_baseline    float64
sbp_week4       float64
sbp_week8       float64
dtype: object


,clinic,age,sbp_baseline,sbp_week4,sbp_week8
patient_id,,,,,
P101,North,58,152.0,146.0,138.0
P102,South,64,146.0,140.0,135.0
P103,West,47,138.0,134.0,131.0
P104,North,71,164.0,NaN,150.0
P105,West,55,158.0,150.0,147.0


clinic          North
age                71
sbp_baseline    164.0
sbp_week4         NaN
sbp_week8       150.0
Name: P104, dtype: object

,clinic,age,sbp_baseline,sbp_week4,sbp_week8
patient_id,,,,,
P101,North,58,152.0,146.0,138.0
P102,South,64,146.0,140.0,135.0
P103,West,47,138.0,134.0,131.0


## Task 3: Summarize the readings

### 3.1 Summarize each visit

> **Checkpoint: `output/visit_summary.csv`**
> Three rows, `sbp_baseline`, `sbp_week4`, and `sbp_week8`, under the header `visit,mean,median,count`.

In [ ]:
readings = bp[["sbp_baseline", "sbp_week4", "sbp_week8"]]

visit_summary = None  # TODO: a DataFrame with the columns mean, median, and count, one row per visit
# TODO: name the index visit

display(visit_summary)

highest_baseline = None  # TODO: the patient_id with the highest sbp_baseline
baseline_week8_r = None  # TODO: the correlation of sbp_baseline with sbp_week8
print("highest baseline:", highest_baseline)
print("baseline vs week 8 r:", baseline_week8_r)

# TODO: save visit_summary to SUMMARY_PATH, keeping its visit index

### 3.2 Count patients per clinic

> **Checkpoint: `output/clinic_counts.csv`**
> Four rows under the header `clinic,count`, North first.

In [ ]:
clinic_counts = None  # TODO: the number of patients at each clinic, with value_counts()
display(clinic_counts)
print("clinics:", None)  # TODO: replace None with the number of distinct clinics, from nunique()

# TODO: save clinic_counts to COUNTS_PATH

## Task 4: Build the follow-up list

### 4.1 Select the program patients

In [ ]:
in_program = None  # TODO: True for North or East patients (isin) whose sbp_baseline is 140 or more
followup = None  # TODO: the in_program rows of bp, without the age column

print("program patients:", in_program.sum())
followup

### 4.2 Add the derived columns

In [ ]:
# TODO: add sbp_mean, the mean of each patient's three readings (axis="columns")
# TODO: add change_week8, sbp_week8 minus sbp_baseline

followup

### 4.3 Sort, rank, and save

> **Checkpoint: `output/followup_priority.csv` and `output/followup_priority.parquet`**
> Seven patients, P110 first and P107 last, with the columns `README.md` lists.

In [ ]:
# TODO: sort followup by change_week8, then patient_id, and assign the result back to followup
# TODO: add improvement_rank, the rank of change_week8 with method="min"
display(followup)

# TODO: save followup to FOLLOWUP_PATH, keeping its patient_id index
# TODO: save followup to PARQUET_PATH with to_parquet(), keeping its index (no index=False)

back = None  # TODO: read PARQUET_PATH back with pd.read_parquet()
print("Parquet round trip matches:", back.equals(followup))

## Task 5: Compare clinic and home readings

> **Checkpoint: `output/white_coat_gap.csv`**
> 15 rows under the header `patient_id,white_coat_gap_mmhg`, five of them with a number.

In [ ]:
home = None  # TODO: read HOME_PATH with patient_id as the index

white_coat_gap = None  # TODO: bp's sbp_week8 minus home's home_sbp_week8
# TODO: name the Series white_coat_gap_mmhg

display(white_coat_gap)
print("patients with both readings:", white_coat_gap.count())

# TODO: save white_coat_gap to GAP_PATH, keeping its patient_id index

## Fresh-run check

Click **Restart**, then **Run All**. The last cell prints your score and what to fix.

In [ ]:
from check_assignment import run_checks

_ = run_checks()